# Train Decision Segmenter — CausaGanha

Fine-tunes `neuralmind/bert-base-portuguese-cased` to **identify and segment** the three structural parts of a Brazilian judicial decision:

| Label | Section | Description |
|---|---|---|
| `RELATORIO` | Relatório | Case history and facts summary |
| `FUNDAMENTACAO` | Fundamentação | Legal reasoning |
| `DISPOSITIVO` | Dispositivo | Operative ruling (the actual decision) |

> **Runtime**: GPU (T4 recommended). Enable via Runtime → Change runtime type.

## 1. Setup — clone repo & install deps

In [ ]:
REPO_URL  = "https://github.com/franklinbaldo/causaganha.git"
BRANCH    = "feat/embedder-smart-truncate-and-privacy-dataset-v2"
REPO_DIR  = "/content/causaganha"


In [ ]:
import os
if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")


In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = f"/root/.local/bin:{os.environ['PATH']}"
!uv --version


In [ ]:
!uv pip install --system -e ".[embeddings]" transformers accelerate datasets seqeval


## 2. Download data from Internet Archive

In [ ]:
import os, urllib.request

PARQUET_DIR = f"{REPO_DIR}/data/test_parquets"
os.makedirs(PARQUET_DIR, exist_ok=True)

FILES = {
    "textos.parquet": "https://archive.org/download/causaganha-test-parquets/textos.parquet",
}

for fname, url in FILES.items():
    dest = os.path.join(PARQUET_DIR, fname)
    if os.path.exists(dest):
        print(f"  Already exists: {fname}")
        continue
    print(f"  Downloading {fname} ...")
    urllib.request.urlretrieve(url, dest)
    size = os.path.getsize(dest)
    print(f"  OK {fname} ({size:,} bytes)")

print('All files ready.')


## 3. Prepare labeled dataset

Uses heuristic markers (`ante o exposto`, `fundamentação`, etc.) to label each document's paragraphs. These are **silver labels** — good enough to bootstrap a classifier that will outperform the heuristic on unseen text.

In [ ]:
import sys
sys.path.insert(0, f"{REPO_DIR}/src")

import re, json, random
import ibis
from pathlib import Path

_DISPOSITIVO_RE = re.compile(
    r'(?:ante\s+o\s+exposto|posto\s+isso|isso\s+posto|'
    r'diante\s+do\s+exposto|pelo\s+exposto|em\s+face\s+do\s+exposto|'
    r'por\s+tais\s+fundamentos|nestes\s+termos|em\s+conclus[\u00e3a]o|'
    r'pelo\s+que\s+exposto|em\s+vista\s+do\s+exposto)',
    re.IGNORECASE,
)
_FUNDAMENTACAO_RE = re.compile(
    r'(?:fundament[ao](?:\u00e7\u00e3o)?|m[\u00e9e]rito|an[\u00e1a]lise\s+do\s+pedido|'
    r'da\s+an[\u00e1a]lise|do\s+m[\u00e9e]rito|'
    r'fundamenta[\u00e7c][\u00e3a]o\s+(?:jur[\u00edi]dica|do\s+ju[\u00edi]zo))',
    re.IGNORECASE,
)

def split_paragraphs(text):
    """Split text into (start, end, paragraph) tuples."""
    paras = []
    pos = 0
    for para in re.split(r'\n{2,}', text):
        if para.strip():
            paras.append((pos, pos + len(para), para))
        pos += len(para) + 2  # account for split separator
    return paras

def label_document(text):
    """Label each paragraph: 0=relatorio, 1=fundamentacao, 2=dispositivo."""
    disp_m = _DISPOSITIVO_RE.search(text)
    if not disp_m:
        return None
    disp_start = disp_m.start()
    pre_disp = text[:disp_start]
    fund_m = _FUNDAMENTACAO_RE.search(pre_disp)
    fund_start = fund_m.start() if fund_m else len(pre_disp) // 2

    paras = split_paragraphs(text)
    labeled = []
    for start, end, para in paras:
        mid = (start + end) / 2
        if mid < fund_start:
            label = 0  # RELATORIO
        elif mid < disp_start:
            label = 1  # FUNDAMENTACAO
        else:
            label = 2  # DISPOSITIVO
        labeled.append({'text': para, 'label': label})
    return labeled

t = ibis.read_parquet(Path(PARQUET_DIR) / 'textos.parquet')
df = t.filter(t.texto.notnull()).execute()
print(f'Loaded {len(df):,} documents')

all_paras = []
skipped = 0
for _, row in df.iterrows():
    labeled = label_document(row['texto'])
    if labeled is None:
        skipped += 1
        continue
    all_paras.extend(labeled)

print(f'Paragraphs: {len(all_paras):,} (skipped {skipped} docs without dispositivo)')
label_counts = {0: 0, 1: 0, 2: 0}
for p in all_paras:
    label_counts[p['label']] += 1
labels_map = {0: 'RELATORIO', 1: 'FUNDAMENTACAO', 2: 'DISPOSITIVO'}
for k, v in sorted(label_counts.items()):
    print(f'  {labels_map[k]}: {v:,}')


## 4. Build HuggingFace Dataset

In [ ]:
from datasets import Dataset
import random

random.seed(42)
random.shuffle(all_paras)

n = len(all_paras)
train_end = int(n * 0.8)
val_end   = train_end + int(n * 0.1)

train_ds = Dataset.from_list(all_paras[:train_end])
val_ds   = Dataset.from_list(all_paras[train_end:val_end])
test_ds  = Dataset.from_list(all_paras[val_end:])

print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}')


## 5. Tokenize

Using `neuralmind/bert-base-portuguese-cased` — 110M params, trained on Brazilian Portuguese corpora, strong baseline for legal PT-BR.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    # No padding here — DataCollatorWithPadding handles dynamic padding per batch
    return tokenizer(batch['text'], truncation=True, max_length=512)

train_ds = train_ds.map(tokenize, batched=True, remove_columns=['text'])
val_ds   = val_ds.map(tokenize,   batched=True, remove_columns=['text'])
test_ds  = test_ds.map(tokenize,  batched=True, remove_columns=['text'])

train_ds = train_ds.rename_column('label', 'labels')
val_ds   = val_ds.rename_column('label',   'labels')
test_ds  = test_ds.rename_column('label',  'labels')

# Note: no set_format('torch') — it triggers a torchvision.VideoReader import bug
# in Colab. DataCollatorWithPadding handles tensor conversion correctly.
print('Tokenization done.')


## 6. Fine-tune

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
import numpy as np
from sklearn.metrics import classification_report

ID2LABEL = {0: 'RELATORIO', 1: 'FUNDAMENTACAO', 2: 'DISPOSITIVO'}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    report = classification_report(
        labels, preds,
        target_names=list(ID2LABEL.values()),
        output_dict=True,
        zero_division=0,
    )
    return {
        'accuracy': report['accuracy'],
        'f1_relatorio':     report['RELATORIO']['f1-score'],
        'f1_fundamentacao': report['FUNDAMENTACAO']['f1-score'],
        'f1_dispositivo':   report['DISPOSITIVO']['f1-score'],
        'macro_f1':         report['macro avg']['f1-score'],
    }

# warmup_steps replaces deprecated warmup_ratio
total_steps = (len(train_ds) // 16) * 3
warmup_steps = max(50, total_steps // 10)

training_args = TrainingArguments(
    output_dir="/content/decision_segmenter",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    fp16=True,
    logging_steps=50,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()


## 7. Evaluate on test set

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

preds_out = trainer.predict(test_ds)
preds     = np.argmax(preds_out.predictions, axis=-1)
labels    = preds_out.label_ids

print(classification_report(
    labels, preds,
    target_names=['RELATORIO', 'FUNDAMENTACAO', 'DISPOSITIVO'],
    zero_division=0,
))


## 8. Save & download model

In [ ]:
import shutil
from google.colab import files

MODEL_OUT = "/content/decision_segmenter_best"
trainer.save_model(MODEL_OUT)
tokenizer.save_pretrained(MODEL_OUT)
print(f"Model saved to {MODEL_OUT}")

shutil.make_archive('/content/decision_segmenter', 'zip', MODEL_OUT)
files.download('/content/decision_segmenter.zip')
print('Downloaded decision_segmenter.zip')
